In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import sys

# Ruta compartida de la base de datos del proyecto
RUTA = Path("data") / "ML.dta"

# Verificar acceso a la base de datos
print("Python:", sys.executable)
print("¿Existe la base?:", RUTA.is_file())

if RUTA.is_file():
    print("Tamaño (GB):", round(RUTA.stat().st_size / 1024**3, 3))
    print("Base de datos localizada correctamente.")
else:
    print("La base todavía no está disponible en la carpeta data.")

Python: C:\Users\SEBASTIAN PC\Proyecto_DSA\.venv\Scripts\python.exe
¿Existe la base?: True
Tamaño (GB): 1.078
Base de datos localizada correctamente.


In [8]:
# Leer una muestra inicial de la base de datos

df_muestra = pd.read_stata(
    RUTA,
    convert_categoricals=False,
    chunksize=1000
)

df_muestra = next(df_muestra)

# Revisar dimensiones y estructura
print("Dimensiones de la muestra:", df_muestra.shape)
print("\nNúmero de variables:", len(df_muestra.columns))

# Mostrar nombres y tipos de variables
display(pd.DataFrame({
    "Variable": df_muestra.columns,
    "Tipo de dato": df_muestra.dtypes.astype(str).values
}))

Dimensiones de la muestra: (1000, 52)

Número de variables: 52


,Variable,Tipo de dato
0,id_anon,float32
1,year,int16
2,period,int16
3,female,int8
4,age,int8
5,student_stratum,int8
6,mothers_education,int8
7,math_score,float64
8,math_score_std,float32
9,school_code_dane,float64


In [9]:
# Revisar la estructura de la muestra inicial

print("Número de filas:", df_muestra.shape[0])
print("Número de columnas:", df_muestra.shape[1])

# Crear tabla con nombres y tipos de variables
tabla_variables = pd.DataFrame({
    "Variable": df_muestra.columns,
    "Tipo de dato": df_muestra.dtypes.astype(str).values,
    "Nulos en la muestra": df_muestra.isna().sum().values
})

display(tabla_variables)

Número de filas: 1000
Número de columnas: 52


,Variable,Tipo de dato,Nulos en la muestra
0,id_anon,float32,0
1,year,int16,0
2,period,int16,0
3,female,int8,0
4,age,int8,0
5,student_stratum,int8,0
6,mothers_education,int8,0
7,math_score,float64,0
8,math_score_std,float32,0
9,school_code_dane,float64,0


In [10]:
# Distribución de la variable objetivo

print("Distribución de la variable hed:")
print(df_muestra["hed"].value_counts(dropna=False))

print("\nPorcentajes:")
print(
    df_muestra["hed"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)

# Comparar con los valores faltantes de las variables posteriores al ingreso
print("\nRelación entre hed y los nulos de ies1:")
print(
    pd.crosstab(
        df_muestra["hed"],
        df_muestra["ies1"].isna(),
        rownames=["hed"],
        colnames=["ies1 es nulo"]
    )
)

Distribución de la variable hed:
hed
0    748
1    252
Name: count, dtype: int64

Porcentajes:
hed
0    74.8
1    25.2
Name: proportion, dtype: float64

Relación entre hed y los nulos de ies1:
ies1 es nulo  False  True 
hed                       
0                 0    748
1               252      0


In [11]:
# Análisis general de la base completa por bloques

total_filas = 0
conteo_hed = pd.Series(dtype="int64")
conteo_nulos = pd.Series(dtype="int64")

reader = pd.read_stata(
    RUTA,
    chunksize=100_000,
    convert_categoricals=False
)

for bloque in reader:

    # Contar registros
    total_filas += len(bloque)

    # Contar valores de la variable objetivo
    conteo_hed = conteo_hed.add(
        bloque["hed"].value_counts(dropna=False),
        fill_value=0
    )

    # Contar valores nulos por variable
    conteo_nulos = conteo_nulos.add(
        bloque.isna().sum(),
        fill_value=0
    )

# Resultados generales
print("Total de registros:", total_filas)

print("\nDistribución de hed:")
print(conteo_hed.astype(int))

print("\nPorcentaje de hed:")
print((conteo_hed / total_filas * 100).round(2))

# Tabla de valores faltantes
tabla_nulos = pd.DataFrame({
    "Variable": conteo_nulos.index,
    "Nulos": conteo_nulos.astype(int).values,
    "Porcentaje": (conteo_nulos / total_filas * 100).round(2).values
})

tabla_nulos = tabla_nulos.sort_values(
    "Porcentaje", ascending=False
).reset_index(drop=True)

display(tabla_nulos)

Total de registros: 8154107

Distribución de hed:
hed
0    4274120
1    3879987
dtype: int64

Porcentaje de hed:
hed
0    52.42
1    47.58
dtype: float64


,Variable,Nulos,Porcentaje
0,ies_university_tech,4274120,52.42
1,id_mun_programa1,4274120,52.42
2,ies_university,4274120,52.42
3,enroll1,4274120,52.42
4,ies_technical_tech,4274120,52.42
5,ies1,4274120,52.42
6,sector_private,4274120,52.42
7,program_inperson,4274120,52.42
8,year_activo1,4274120,52.42
9,year_enrolls1,4274120,52.42


In [12]:
# Analizar ingreso a educación superior por estrato


resultados = []

reader = pd.read_stata(
    RUTA,
    columns=["student_stratum", "hed"],
    chunksize=100_000,
    convert_categoricals=False
)

for bloque in reader:

    resumen = bloque.groupby(
        "student_stratum", dropna=False
    )["hed"].agg(["count", "sum"])

    resultados.append(resumen)

# Consolidar los resultados de todos los bloques
tabla_estrato = pd.concat(resultados).groupby(level=0, dropna=False).sum()

tabla_estrato["Tasa ingreso (%)"] = (
    tabla_estrato["sum"] / tabla_estrato["count"] * 100
).round(2)

tabla_estrato = tabla_estrato.rename(columns={
    "count": "Estudiantes",
    "sum": "Ingresaron"
})

display(tabla_estrato)

,Estudiantes,Ingresaron,Tasa ingreso (%)
student_stratum,,,
1.0,2508623,800902,31.93
2.0,3196457,1485491,46.47
3.0,1763807,1097800,62.24
4.0,315289,247368,78.46
5.0,122673,96960,79.04
6.0,71502,55129,77.10
NaN,175756,96337,54.81


In [13]:
# Analizar ingreso a educación superior por tipo de colegio

resultados = []

reader = pd.read_stata(
    RUTA,
    columns=["official_school", "hed"],
    chunksize=100_000,
    convert_categoricals=False
)

for bloque in reader:
    resumen = bloque.groupby(
        "official_school", dropna=False
    )["hed"].agg(["count", "sum"])

    resultados.append(resumen)

tabla_colegio = pd.concat(resultados).groupby(level=0, dropna=False).sum()

tabla_colegio["Tasa ingreso (%)"] = (
    tabla_colegio["sum"] / tabla_colegio["count"] * 100
).round(2)

tabla_colegio = tabla_colegio.rename(columns={
    "count": "Estudiantes",
    "sum": "Ingresaron"
})

display(tabla_colegio)

,Estudiantes,Ingresaron,Tasa ingreso (%)
official_school,,,
0,2410927,1504643,62.41
1,5743180,2375344,41.36


In [14]:
# Analizar la tasa de ingreso a educación superior por año

resultados = []

reader = pd.read_stata(
    RUTA,
    columns=["year", "hed"],
    chunksize=100_000,
    convert_categoricals=False
)

for bloque in reader:

    resumen = bloque.groupby("year")["hed"].agg(["count", "sum"])

    resultados.append(resumen)

# Consolidar los resultados
tabla_anios = pd.concat(resultados).groupby(level=0).sum()

tabla_anios["Tasa ingreso (%)"] = (
    tabla_anios["sum"] / tabla_anios["count"] * 100
).round(2)

tabla_anios = tabla_anios.rename(columns={
    "count": "Estudiantes",
    "sum": "Ingresaron"
})

display(tabla_anios)

,Estudiantes,Ingresaron,Tasa ingreso (%)
year,,,
2000,343036,117534,34.26
2001,317568,117736,37.07
2002,312492,127890,40.93
2003,324971,133829,41.18
2004,321439,140243,43.63
2005,352964,156357,44.30
2006,370633,167892,45.30
2007,401838,182869,45.51
2008,404405,187989,46.49


In [15]:
# Revisar la ventana de seguimiento por cohorte

resultados = []

reader = pd.read_stata(
    RUTA,
    columns=["year", "hed", "window_hed"],
    chunksize=100_000,
    convert_categoricals=False
)

for bloque in reader:

    resumen = bloque.groupby("year").agg(
        estudiantes=("hed", "size"),
        ingresaron=("hed", "sum"),
        window_no_nulos=("window_hed", "count")
    )

    resultados.append(resumen)

tabla_ventana = pd.concat(resultados).groupby(level=0).sum()

tabla_ventana["tasa_ingreso"] = (
    tabla_ventana["ingresaron"] /
    tabla_ventana["estudiantes"] * 100
).round(2)

tabla_ventana["pct_window_observado"] = (
    tabla_ventana["window_no_nulos"] /
    tabla_ventana["estudiantes"] * 100
).round(2)

display(tabla_ventana)

,estudiantes,ingresaron,window_no_nulos,tasa_ingreso,pct_window_observado
year,,,,,
2000,343036,117534,117534,34.26,34.26
2001,317568,117736,117736,37.07,37.07
2002,312492,127890,127890,40.93,40.93
2003,324971,133829,133829,41.18,41.18
2004,321439,140243,140243,43.63,43.63
2005,352964,156357,156357,44.30,44.30
2006,370633,167892,167892,45.30,45.30
2007,401838,182869,182869,45.51,45.51
2008,404405,187989,187989,46.49,46.49


In [16]:
# Distribución de ingreso y no ingreso por año

tabla_anios["No ingresaron"] = (
    tabla_anios["Estudiantes"] - tabla_anios["Ingresaron"]
)

tabla_anios["Tasa no ingreso (%)"] = (
    tabla_anios["No ingresaron"] /
    tabla_anios["Estudiantes"] * 100
).round(2)

display(tabla_anios[
    ["Estudiantes", "Ingresaron",
     "No ingresaron", "Tasa ingreso (%)",
     "Tasa no ingreso (%)"]
].tail(6))

,Estudiantes,Ingresaron,No ingresaron,Tasa ingreso (%),Tasa no ingreso (%)
year,,,,,
2014,471951,248523,223428,52.66,47.34
2015,476220,248840,227380,52.25,47.75
2016,480241,252547,227694,52.59,47.41
2017,455074,234884,220190,51.61,48.39
2018,459657,232746,226911,50.63,49.37
2019,474638,228210,246428,48.08,51.92


In [17]:
# Verificar consistencia entre hed y no_hed

total_inconsistencias = 0

reader = pd.read_stata(
    RUTA,
    columns=["hed", "no_hed"],
    chunksize=100_000,
    convert_categoricals=False
)

for bloque in reader:
    inconsistencias = (
        (bloque["hed"] + bloque["no_hed"]) != 1
    ).sum()

    total_inconsistencias += inconsistencias

print("Registros inconsistentes:", total_inconsistencias)

Registros inconsistentes: 0
